In [1]:
import numpy as np
import cupy as cp
from tanalysis import improcess, stitching, stitching_gpu, traj_analysis
import tifffile as tiff
import os
import matplotlib.pyplot as plt



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.12.0 
torch version:  	2.8.0+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [ ]:
dirn = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals\24h_CXCL10_Conc10_z5_t8h-0.tif"
imgs, names, info = improcess.imread(dirn, tiles=False, gpu=False)
img = imgs[0][22][4]
m,n = img.shape
U, S, V = np.linalg.svd(img)
k = int(0.1*m)
sigma = np.eye(k)*S[:k]
nU = U[:,:k]
nV = V[:k,:]
print(nU.shape, sigma.shape, nV.shape)
nimg = nU@sigma@nV
savedir = r"C:\Users\pcanaleta\Desktop"
plt.imsave(os.path.join(savedir, 'original.png'), img)
plt.imsave(os.path.join(savedir, 'svd.png'), nimg)
#print(U,S,V)

In [ ]:
dirn = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals"
for fname in os.listdir(dirn):
    dirname = os.path.join(dirn, fname)
    imgs, names, info = improcess.imread(dirname, tiles=True, gpu=True)
    positions = info['mosaic_position']
    translations_list = stitching_gpu.translationComputation(imgs, positions, n=20, n_frames=10)
    res_img_list = stitching_gpu.image_reconstruction(imgs, positions, translations_list)
    print(res_img_list[0].shape)
    newname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Results"
    im = cp.asnumpy(res_img_list[0])
        
    for name in names:
        tiff.imwrite(
            fr'{newname}\{name}.tif', 
            im[:,:,:,:].astype(np.uint8), 
            imagej=True,
            metadata={
                'axes':'TZYX'
            })


In [2]:
dname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD1.T\EXP.HD1.15.6\EXP.HD1.15.6.1_Processed\24h\Originals"
model = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\training_process\processed_imgs\models\processed_imgs"

for fname in os.listdir(dname):
    dirname = os.path.join(dname, fname)
    imgs, names, info = improcess.imread(dirname)
    savedir = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD1.T\EXP.HD1.15.6\EXP.HD1.15.6.1_Processed\24h\Results"
    temp_savedir = improcess.cellposeseg(imgs, 3, names, savedir, modelpath=model)

Reading submitted files: 100%|████████████████████████████████████████| 1/1 [00:01<00:00,  1.21s/it]

All files read!
2025-10-20 11:00:07,547 [INFO] WRITING LOG OUTPUT TO C:\Users\pcanaleta\.cellpose\run.log
2025-10-20 11:00:07,548 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.12.0 
torch version:  	2.8.0+cu126
2025-10-20 11:00:07,628 [INFO] ** TORCH CUDA version installed and working. **
2025-10-20 11:00:07,629 [INFO] >>>> using GPU (CUDA)


2025-10-20 11:00:08,565 [INFO] >>>> loading model C:\Users\pcanaleta\.cellpose\models\cpsam
2025-10-20 11:00:08,968 [INFO] ** TORCH CUDA version installed and working. **
2025-10-20 11:00:08,968 [INFO] >>>> using GPU (CUDA)
2025-10-20 11:00:09,766 [INFO] >>>> loading model C:\Users\pcanaleta\Documents\Cellpose_segmentation\training_process\processed_imgs\models\processed_imgs
C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD1.T\EXP.HD1.15.6\EXP.HD1.15.6.1_Processed\24h\Results\WT_2_2_median_Bsubs-0
2025-10-20 11:00:10,782 [INFO] running YX: 40 planes of size (1024, 1024)
2025-10-20 11:00:37,624 [INFO] 100%|##########| 40/40 [00:26<00:00,  1.49it/s]
2025-10-20 11:00:37,873 [INFO] running ZY: 1024 planes of size (40, 1024)
2025-10-20 11:03:08,790 [INFO] 100%|##########| 1024/1024 [02:30<00:00,  6.79it/s]
2025-10-20 11:03:09,048 [INFO] running ZX: 1024 planes of size (40, 1024)
2025-10-20 11:05:41,451 [INFO] 100%|##########| 1024/1024 [02:32<00:00,  6.72it/s]
2025-10-20 11:05:43,0